# MxCy Physics-Constrained ML Pipeline — Version 6.0
### Transition-Metal Chalcogenides: Band Gap, Magnetization & Half-Metal Prediction

**Based on:** Reggad *et al.*, *Physica B* **526** (2017) 89–95  
**and:** Reggad, *Thèse de Doctorat*, Université Sidi-Bel-Abbès

> **v6.0 changes (on top of v5.1):**
>
> | Category | Change | Reason |
> |---|---|---|
> | 🔴 Bug fix | Magnetization model rebuilt — ordering encoded as *feature*, not as split | Sub-model split on ~226 samples → R²(mag) = −0.75 in v5.1 |
> | 🔴 Bug fix | `gap_log_proxy` removed from feature set | Partially circular with gap target → inflated importance |
> | ✨ ML | XGBoost added to all ensembles | Superior gradient boosting for tabular physics data |
> | ✨ ML | Nested CV for hyperparameter tuning | Prevents optimistic bias from tuning on validation fold |
> | ✨ ML | Bootstrap conformal prediction intervals | 90 % calibrated uncertainty on gap and magnetization |
> | ✨ Physics | 5 new features → 27 total | ZSA charge-transfer energy, SOC², Stoner exchange splitting, d-filling ratio, Goldschmidt tolerance |
> | ✨ Data | Paper database ingestion module | Extract band gap / magnetization from your PDF library |
> | ✨ Data | Google Drive PDF reader (Cell 6b) | Load and parse chalcogenide papers directly from Drive |

---
## 📊 Version comparison (key metrics)

| Metric | v4.0 | v5.1 | **v6.0 target** |
|--------|------|------|-----------------|
| R²(gap) CV | 0.066 | 0.353 | **≥ 0.55** |
| R²(gap) test | 0.517 | 0.563 | **≥ 0.65** |
| MAE gap (eV) | 0.261 | 0.115 | **≤ 0.09** |
| R²(mag) test | 0.718 | **−0.754** | **≥ 0.75** |
| Ordering acc | 77.4 % | 78.6 % | **≥ 82 %** |
| Features | 9 | 23 | **27** |

---
## 🗺️ How to Run

| Platform | Steps |
|----------|-------|
| **Kaggle** | Upload → Settings → Secrets → `MP_API_KEY` → Run Cell 1 → Restart & Clear → Run All |
| **Colab** | Upload → Run Cell 1 (auto-restarts) → paste key in Cell 3 → Run All from Cell 2 |
| **Local** | `export MP_API_KEY=<key>` → `jupyter nbconvert --to notebook --execute` |


## Cell 1 — Setup
> ⚠️ **Run this cell ALONE, then restart the kernel before running anything else.**

In [ ]:
import sys, subprocess, os

def _detect_platform():
    try:
        import google.colab; return "colab"
    except ImportError:
        pass
    return "kaggle" if os.path.exists("/kaggle") else "local"

PLATFORM = _detect_platform()
print(f"▶  Platform detected: {PLATFORM.upper()}")

def _packages_ok():
    try:
        import mp_api, pyarrow as pa, shap, openpyxl, xgboost
        major, minor = (int(x) for x in pa.__version__.split(".")[:2])
        return (major, minor) >= (16, 0)
    except Exception:
        return False

if _packages_ok():
    print("✅ All packages already installed — skipping.")
else:
    _PACKAGES = [
        "mp-api", "pyarrow>=16.0.0", "shap", "openpyxl",
        "xgboost>=2.0.0",          # v6.0 new
        "requests==2.32.4",
    ]
    print("📦 Installing packages (mp-api may take 2–4 min) …")
    _ipy = get_ipython() if "get_ipython" in dir() else None
    if _ipy:
        _ipy.run_line_magic("pip", "install " + " ".join(f'"{p}"' for p in _PACKAGES) + " --progress-bar off")
    else:
        subprocess.check_call([sys.executable, "-m", "pip", "install", *_PACKAGES])
    print("✅ Done.")
    if PLATFORM == "colab":
        import os; os.kill(os.getpid(), 9)
    else:
        print("⚠️  Restart your kernel manually if pyarrow/xgboost was upgraded.")


## Cell 2 — Imports & Environment
> ✅ Start here after kernel restart.

In [ ]:
import sys, os, math, warnings, joblib, copy, json
from itertools import product

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from openpyxl import Workbook
from openpyxl.utils.dataframe import dataframe_to_rows
from sklearn.ensemble import (RandomForestRegressor, RandomForestClassifier,
                               GradientBoostingRegressor,
                               HistGradientBoostingRegressor,
                               ExtraTreesRegressor)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import (KFold, StratifiedKFold,
                                      StratifiedShuffleSplit,
                                      RandomizedSearchCV)
from sklearn.calibration import CalibratedClassifierCV
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import r2_score, mean_absolute_error, accuracy_score
from sklearn.multioutput import MultiOutputRegressor
import xgboost as xgb

try:
    import shap
    SHAP_OK = True
except ImportError:
    SHAP_OK = False
    print("⚠️  shap not available — SHAP analysis will be skipped")

from mp_api.client import MPRester

warnings.filterwarnings("ignore")
print("✅ All imports OK.")
print(f"   xgboost {xgb.__version__}  |  sklearn OK  |  shap {'OK' if SHAP_OK else 'missing'}")


## Cell 3 — Platform Detection & API Key

In [ ]:
def _detect_platform():
    try:
        import google.colab; return "colab"
    except ImportError:
        pass
    return "kaggle" if os.path.exists("/kaggle") else "local"

PLATFORM = _detect_platform()
print(f"▶  Platform (post-restart): {PLATFORM.upper()}")

if PLATFORM == "kaggle":
    OUTPUT_DIR = "/kaggle/working"
elif PLATFORM == "colab":
    OUTPUT_DIR = "/content/MxCy_outputs_v6"
else:
    OUTPUT_DIR = "./MxCy_outputs_v6"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"📁 Output directory: {OUTPUT_DIR}")

# ── API key ───────────────────────────────────────────────────────────────
# Paste your Materials Project API key between the quotes:
_HARDCODED_KEY = ""   # ← e.g. "AbCdEf1234567890"

MP_API_KEY = _HARDCODED_KEY if len(_HARDCODED_KEY) >= 10 else os.environ.get("MP_API_KEY", "")
if MP_API_KEY and len(MP_API_KEY) >= 10:
    print(f"🔑 MP_API_KEY ready ({len(MP_API_KEY)} chars) ✅")
else:
    print("⚠️  MP_API_KEY not set — paste your key into _HARDCODED_KEY above.")


## Cell 4 — Atomic & Physics Constants *(v6.0 — 27-feature set)*

### New features in v6.0

| Feature | Formula | Physics |
|---|---|---|
| `zsa_delta` | Δ = ε_d − ε_p − U/2 | Zaanen-Sawatzky-Allen charge-transfer energy; Δ < 0 → charge-transfer insulator (O, S) |
| `soc_sq` | λ² | Second-order SOC → orbital quenching in 4d/5d |
| `stoner_split` | I × W | Direct Stoner exchange splitting; > 1 → spontaneous magnetization |
| `d_fill_ratio` | n_d / 10 | Normalised d-filling (0 → 1); half-filled = 0.5 |
| `goldschmidt_t` | (r_M + r_C) / (√2 (r_M + r_C)) | Structure stability / distortion proxy |

Note: `gap_log_proxy` **removed** from v5.1 (partially circular with target).


In [ ]:
# ══════════════════════════════════════════════════════════════
#  Atomic & Physics Constants  (v6.0: 27-feature set)
# ══════════════════════════════════════════════════════════════

ATOMIC_DATA = {
    # 3d:  [n_d, r_cov, chi, IE, U_Hubbard, r_ionic]
    "Ti": [2,  1.40, 1.54, 6.82, 3.0, 0.86],
    "V":  [3,  1.35, 1.63, 6.74, 3.5, 0.79],
    "Cr": [4,  1.28, 1.66, 6.77, 3.5, 0.87],
    "Mn": [5,  1.27, 1.55, 7.43, 4.0, 0.97],
    "Fe": [6,  1.26, 1.83, 7.90, 4.5, 0.92],
    "Co": [7,  1.25, 1.88, 7.88, 5.0, 0.88],
    "Ni": [8,  1.24, 1.91, 7.64, 5.5, 0.83],
    "Cu": [9,  1.28, 1.90, 7.73, 7.0, 0.87],
    "Zn": [10, 1.22, 1.65, 9.39, 0.0, 0.88],
    # 4d
    "Zr": [2,  1.75, 1.33, 6.63, 2.0, 0.86],
    "Nb": [3,  1.64, 1.60, 6.76, 2.5, 0.78],
    "Mo": [4,  1.54, 2.16, 7.09, 2.5, 0.83],
    "Ru": [6,  1.46, 2.20, 7.36, 3.0, 0.82],
    "Rh": [7,  1.42, 2.28, 7.46, 3.0, 0.80],
    "Pd": [8,  1.39, 2.20, 8.34, 3.5, 0.78],
    "Ag": [9,  1.45, 1.93, 7.58, 6.0, 1.29],
    # 5d
    "Hf": [2,  1.75, 1.30, 6.83, 2.0, 0.85],
    "Ta": [3,  1.70, 1.50, 7.55, 2.5, 0.78],
    "W":  [4,  1.62, 2.36, 7.86, 2.5, 0.80],
    "Re": [5,  1.51, 1.90, 7.83, 3.0, 0.77],
    "Os": [6,  1.44, 2.20, 8.44, 3.0, 0.77],
    "Ir": [7,  1.41, 2.20, 8.97, 3.0, 0.77],
    "Pt": [8,  1.36, 2.28, 8.96, 3.5, 0.77],
    # Anions: [valence_e, r_cov, chi, r_ionic, polarizability]
    "O":  [6, 0.73, 3.44, 1.40, 0.80],
    "S":  [6, 1.04, 2.58, 1.84, 2.90],
    "Se": [6, 1.17, 2.55, 1.98, 3.77],
    "Te": [6, 1.37, 2.10, 2.21, 5.50],
}

SOC_LAMBDA = {
    "Ti": 20,  "V": 55,   "Cr": 60,  "Mn": 50,  "Fe": 50,  "Co": 88,
    "Ni": 100, "Cu": 110, "Zn": 0,
    "Zr": 100, "Nb": 150, "Mo": 200, "Ru": 300, "Rh": 350, "Pd": 400, "Ag": 200,
    "Hf": 400, "Ta": 600, "W": 800,  "Re": 1000,"Os": 1200,"Ir": 1500,"Pt": 1800,
}

STONER_I = {
    "Ti": 0.58, "V": 0.70, "Cr": 0.77, "Mn": 0.89, "Fe": 0.93, "Co": 0.99,
    "Ni": 1.01, "Cu": 0.73, "Zn": 0.0,
    "Zr": 0.33, "Nb": 0.38, "Mo": 0.44, "Ru": 0.50, "Rh": 0.53, "Pd": 0.56, "Ag": 0.38,
    "Hf": 0.30, "Ta": 0.33, "W": 0.38, "Re": 0.42, "Os": 0.46, "Ir": 0.48, "Pt": 0.50,
}

MADELUNG = {
    "NiAs": 1.560, "rock-salt": 1.748, "zincblende": 1.638,
    "wurtzite": 1.641, "MnP": 1.560, "pyrite": 1.800,
}

# d-band energy (eV) relative to anion p-band — for ZSA Δ
# Source: Bocquet et al. PRB 1996 / Zaanen et al. PRL 1985
D_BAND_ENERGY = {
    "Ti": -1.5, "V": -2.0, "Cr": -2.5, "Mn": -3.0, "Fe": -3.5,
    "Co": -4.0, "Ni": -4.5, "Cu": -5.5, "Zn": -7.0,
    "Zr": -1.0, "Nb": -1.5, "Mo": -2.0, "Ru": -3.0, "Rh": -3.5,
    "Pd": -4.0, "Ag": -5.0, "Hf": -0.8, "Ta": -1.2, "W": -1.8,
    "Re": -2.5, "Os": -3.0, "Ir": -3.5, "Pt": -4.0,
}

# Anion p-band energy (eV) — reference O = 0
ANION_P_ENERGY = {"O": 0.0, "S": 2.5, "Se": 3.5, "Te": 5.0}

TM_ELEMENTS = [
    "Ti", "V",  "Cr", "Mn", "Fe", "Co", "Ni", "Cu", "Zn",
    "Zr", "Nb", "Mo", "Ru", "Rh", "Pd", "Ag",
    "Hf", "Ta", "W",  "Re", "Os", "Ir", "Pt",
]

CHALCOGENS = ["O", "S", "Se", "Te"]

STOICHIOMETRIES = {
    "MC_NiAs":   {"structure": "NiAs",       "x": 1, "y": 1, "angle": 165.0},
    "MC_NaCl":   {"structure": "rock-salt",  "x": 1, "y": 1, "angle": 180.0},
    "MC_ZB":     {"structure": "zincblende", "x": 1, "y": 1, "angle": 109.5},
    "MC_WZ":     {"structure": "wurtzite",   "x": 1, "y": 1, "angle": 109.5},
    "MC_MnP":    {"structure": "MnP",        "x": 1, "y": 1, "angle": 155.0},
    "MC2_Pyrite":{"structure": "pyrite",     "x": 1, "y": 2, "angle": 180.0},
    "M2C3_Corun":{"structure": "NiAs",       "x": 2, "y": 3, "angle": 130.0},
    "M2C_AntiF": {"structure": "zincblende", "x": 2, "y": 1, "angle": 109.5},
    "M3C4_Spinel":{"structure":"rock-salt",  "x": 3, "y": 4, "angle": 125.0},
}

# v6 feature list — 27 features (gap_log_proxy removed, 5 new added)
FEATURES_V6 = [
    # ── original 9 ──
    "d_electrons", "Hubbard_U", "chi_diff", "bond_length_A",
    "bandwidth_W", "U_W_ratio", "bond_angle_deg", "ca_ratio",
    "exchange_corr_ratio",
    # ── v5 additions (10) ──
    "nd_half_fill", "r_ionic_ratio", "ionicity", "anion_polarizability",
    "soc_lambda", "crystal_field_10Dq", "jahn_teller_active",
    "superexchange_factor", "valence_electron_count", "madelung_energy",
    # ── binary flags (3) ──
    "is_oxide", "is_3d_metal", "is_5d_metal",
    # ── v6 new (5) ──
    "zsa_delta", "soc_sq", "stoner_split", "d_fill_ratio", "goldschmidt_t",
]

print(f"✅ Atomic constants loaded. Feature set: {len(FEATURES_V6)} features.")
print(f"   TM elements: {len(TM_ELEMENTS)}  |  Chalcogens: {len(CHALCOGENS)}")
print(f"   Stoichiometries: {len(STOICHIOMETRIES)}")


## Cell 5 — Feature Engineering v6.0
*(27 physics-based features — `gap_log_proxy` removed, 5 new ZSA/SOC²/Stoner features added)*

In [ ]:
# ══════════════════════════════════════════════════════════════
#  Feature Engineering — v6.0
# ══════════════════════════════════════════════════════════════

STRUCT_CA = {
    "NiAs": 1.534, "rock-salt": 1.000, "zincblende": 1.000,
    "wurtzite": 1.633, "MnP": 1.580, "pyrite": 1.000,
}

JT_CONFIGS = {1, 4, 7, 9}   # d^1, d^4, d^7, d^9 are JT-active

def compute_all_features_v6(M, C, x, y, structure, bond_angle_deg):
    """
    Compute the full 27-feature vector for compound M_x C_y in given structure.

    v6 additions vs v5.1:
      - zsa_delta    : Zaanen–Sawatzky–Allen charge-transfer energy
      - soc_sq       : λ² (second-order SOC)
      - stoner_split : Stoner I × bandwidth W
      - d_fill_ratio : n_d / 10 (normalised d-filling)
      - goldschmidt_t: ionic-radius-based structure-stability proxy
      - gap_log_proxy REMOVED (circular with target)
    """
    ad_M = ATOMIC_DATA[M]   # [n_d, r_cov, chi, IE, U_Hubbard, r_ionic]
    ad_C = ATOMIC_DATA[C]   # [valence_e, r_cov, chi, r_ionic, polarizability]

    n_d   = ad_M[0];  r_cov_M = ad_M[1]; chi_M = ad_M[2]
    IE_M  = ad_M[3];  U       = ad_M[4]; r_ion_M = ad_M[5]

    val_C = ad_C[0];  r_cov_C = ad_C[1]; chi_C = ad_C[2]
    r_ion_C = ad_C[3]; pol_C = ad_C[4]

    # ── original features ─────────────────────────────────────
    chi_diff  = abs(chi_C - chi_M)
    bond_l    = (r_cov_M + r_cov_C) * (x + y) / (x + y)   # mean cov bond
    # Refined bond length per stoichiometry
    if x == 1 and y == 1:
        bond_l = r_cov_M + r_cov_C
    elif x == 1 and y == 2:
        bond_l = r_cov_M + r_cov_C * 1.05
    elif x == 2 and y == 1:
        bond_l = r_cov_M * 1.05 + r_cov_C
    elif x == 2 and y == 3:
        bond_l = (r_cov_M + r_cov_C) * 1.02
    else:
        bond_l = r_cov_M + r_cov_C

    # Bandwidth W (Harrison model: W ∝ 1/d^5 for d-d, ∝ 1/d^3.5 for d-p)
    W = 3.5 / (bond_l ** 3.5)
    UW = U / (W + 1e-9)

    ca = STRUCT_CA.get(structure, 1.0)

    I_stoner  = STONER_I.get(M, 0.5)
    exch_corr = I_stoner / (W + 1e-9)

    ba_deg    = bond_angle_deg
    ba_rad    = math.radians(ba_deg)

    # superexchange factor (Goodenough-Kanamori)
    sup_f = math.sin(ba_rad/2)**2 * math.cos(ba_rad/2)**2

    # valence electron count
    vec = x * n_d + y * val_C

    # Madelung energy (M·z²/(r_ion_M + r_ion_C))
    mad_c = MADELUNG.get(structure, 1.6)
    z_eff = math.sqrt(abs(n_d - val_C))   # rough effective charge
    mad_e = mad_c * z_eff**2 / (r_ion_M + r_ion_C + 1e-9)

    # crystal field 10Dq (empirical: ∝ z_eff / bond^5)
    cf_10dq = 6.0 * z_eff / (bond_l**5 + 1e-9)

    # JT activity
    jt = int(n_d in JT_CONFIGS)

    # nd_half_fill: closeness to half-filling
    nd_hf = 1.0 - abs(n_d - 5.0) / 5.0

    # ionic radius ratio
    r_ratio = r_ion_M / (r_ion_C + 1e-9)

    # Phillips ionicity
    ionicity = (chi_C - chi_M) / (chi_C + chi_M + 1e-9)

    # SOC lambda
    soc_lam = SOC_LAMBDA.get(M, 0.0)

    # is_ flags
    is_ox  = int(C == "O")
    is_3d  = int(M in ["Ti","V","Cr","Mn","Fe","Co","Ni","Cu","Zn"])
    is_5d  = int(M in ["Hf","Ta","W","Re","Os","Ir","Pt"])

    # ── v6 new features ───────────────────────────────────────
    # 1. ZSA charge-transfer energy: Δ = ε_d - ε_p - U/2
    #    Δ < 0  →  charge-transfer insulator (anion→metal)
    #    Δ > 0  →  Mott-Hubbard insulator (on-site)
    eps_d   = D_BAND_ENERGY.get(M, -3.0)
    eps_p   = ANION_P_ENERGY.get(C, 0.0)
    zsa_d   = eps_d - eps_p - U / 2.0

    # 2. SOC² (second-order perturbation → orbital quenching)
    soc_sq  = soc_lam**2 / 1e6    # normalise to eV² scale

    # 3. Stoner exchange splitting E_ex = I × W
    stoner_split = I_stoner * W

    # 4. Normalised d-filling
    d_fill = n_d / 10.0

    # 5. Goldschmidt tolerance factor (ionic radii proxy for structure stability)
    #    t = (r_M + r_C) / (√2 * (r_M_ref + r_C)) — adapted for binary MX
    r_ref = 1.4   # O²⁻ reference ionic radius
    golds_t = (r_ion_M + r_ion_C) / (math.sqrt(2) * (r_ion_M + r_ref) + 1e-9)

    out = {
        "d_electrons":         float(n_d),
        "Hubbard_U":           float(U),
        "chi_diff":            float(chi_diff),
        "bond_length_A":       float(bond_l),
        "bandwidth_W":         float(W),
        "U_W_ratio":           float(UW),
        "bond_angle_deg":      float(ba_deg),
        "ca_ratio":            float(ca),
        "exchange_corr_ratio": float(exch_corr),
        "nd_half_fill":        float(nd_hf),
        "r_ionic_ratio":       float(r_ratio),
        "ionicity":            float(ionicity),
        "anion_polarizability":float(pol_C),
        "soc_lambda":          float(soc_lam),
        "crystal_field_10Dq":  float(cf_10dq),
        "jahn_teller_active":  float(jt),
        "superexchange_factor":float(sup_f),
        "valence_electron_count": float(vec),
        "madelung_energy":     float(mad_e),
        "is_oxide":            float(is_ox),
        "is_3d_metal":         float(is_3d),
        "is_5d_metal":         float(is_5d),
        # v6 new
        "zsa_delta":           float(zsa_d),
        "soc_sq":              float(soc_sq),
        "stoner_split":        float(stoner_split),
        "d_fill_ratio":        float(d_fill),
        "goldschmidt_t":       float(golds_t),
        # internal (not in FEATURES_V6, used for other calcs)
        "_bl": bond_l, "_W": W, "_UW": UW, "_chi_diff": chi_diff, "_ca": ca,
    }
    return out

print(f"✅ compute_all_features_v6() defined ({len(FEATURES_V6)} features).")


## Cell 6 — Paper Database Integration *(v6.0 new)*

This cell lets you supplement the Materials Project data with experimental or DFT values
from your own papers. Two modes:

**Mode A — CSV table** (fastest): export a table from your papers as CSV  
**Mode B — Google Drive PDF** (see Cell 6b): parse PDFs directly  
**Mode C — Manual dict**: paste values directly below


In [ ]:
# ══════════════════════════════════════════════════════════════
#  Cell 6 — Paper Database Integration
# ══════════════════════════════════════════════════════════════
#
#  Required columns in CSV / dict:
#    compound (str)  e.g. "FeS2"
#    M         (str)  e.g. "Fe"
#    C         (str)  e.g. "S"
#    structure (str)  e.g. "MC2_Pyrite"
#    band_gap_eV (float)
#    magnetization_muB (float)
#    source    (str)  e.g. "Reggad2017"
#
#  Values from papers override MP values only if MP has no entry.
#  If both exist, the MP value is kept (MP-DFT is the reference).

PAPER_DB_CSV_PATH = ""   # ← path to your CSV, e.g. "/content/drive/MyDrive/chalcogenides.csv"

PAPER_DB_MANUAL = [
    # ── Reggad et al., Physica B 526 (2017) 89–95 ──────────────
    # Paste rows from your paper here:
    # {"compound":"FeS2","M":"Fe","C":"S","structure":"MC2_Pyrite",
    #  "band_gap_eV":0.95,"magnetization_muB":0.0,"source":"Reggad2017"},
]

def load_paper_db():
    records = list(PAPER_DB_MANUAL)
    if PAPER_DB_CSV_PATH and os.path.isfile(PAPER_DB_CSV_PATH):
        try:
            df_p = pd.read_csv(PAPER_DB_CSV_PATH)
            required = {"compound","M","C","structure","band_gap_eV","magnetization_muB"}
            if required.issubset(df_p.columns):
                records += df_p.to_dict("records")
                print(f"   📄 Loaded {len(df_p)} rows from CSV: {PAPER_DB_CSV_PATH}")
            else:
                miss = required - set(df_p.columns)
                print(f"   ⚠️  CSV missing columns: {miss}")
        except Exception as e:
            print(f"   ⚠️  Could not read CSV: {e}")
    paper_dict = {}
    for r in records:
        key = (r["M"], r["C"], r["structure"])
        paper_dict[key] = {
            "band_gap_eV":       float(r["band_gap_eV"]),
            "magnetization_muB": float(r["magnetization_muB"]),
            "source":            r.get("source", "paper"),
        }
    print(f"✅ Paper DB: {len(paper_dict)} compound entries loaded.")
    return paper_dict

PAPER_DB = load_paper_db()


## Cell 6b — PDF Directory Reader *(optional)*

Scan a **local or Drive folder** of PDF papers and extract compound data automatically.  
Works on **Windows, Linux, macOS, Colab and Kaggle** — no poppler required (falls back to `pypdf`).  

Set `PAPER_DIR` to the path of the folder containing your PDFs, then run the cell.  
Confirmed rows should be copied into `PAPER_DB_MANUAL` in Cell 6.

In [ ]:
# =================================================================
#  Cell 6b - PDF Directory Reader  (Local / Colab / Kaggle)
# =================================================================
#
#  Set PAPER_DIR to the folder containing your PDF papers.
#  Works on Windows, Linux, macOS, Colab and Kaggle.
#
#  Windows example :  PAPER_DIR = r"G:\MyResearch\AllPapers"
#  Linux / macOS   :  PAPER_DIR = "/home/user/papers"
#  Colab (Drive)   :  PAPER_DIR = "/content/drive/MyDrive/papers"
#  Kaggle          :  PAPER_DIR = "/kaggle/input/my-papers"
#
#  All *.pdf files in that directory are processed automatically.
#  Requires: pypdf  (pip install pypdf).
#  pdftotext (poppler) is used when available for better layout.

from pathlib import Path
import re, subprocess, sys

# -----------------------------------------------------------------
# <- EDIT THIS LINE -----------------------------------------------
PAPER_DIR = r""   # e.g. r"G:\MyResearch\AllPapers"
# -----------------------------------------------------------------


def _maybe_mount_drive(directory):
    """Mount Google Drive on Colab if the path is inside /content/drive."""
    if PLATFORM != "colab":
        return True
    if not str(directory).startswith("/content/drive"):
        return True
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        print("OK Google Drive mounted at /content/drive")
        return True
    except Exception as e:
        print(f"WARNING: Drive mount failed: {e}")
        return False


def _read_pdf_text(pdf_path):
    """
    Extract plain text from *pdf_path* (a pathlib.Path).
    Strategy:
      1. pdftotext -layout  (poppler - best column alignment)
      2. pypdf              (pure-Python fallback, works on Windows)
    Returns an empty string on failure.
    """
    path_str = str(pdf_path)

    # -- attempt 1: pdftotext ----------------------------------------
    try:
        result = subprocess.run(
            ["pdftotext", "-layout", path_str, "-"],
            capture_output=True, text=True, timeout=60
        )
        if result.returncode == 0 and result.stdout.strip():
            return result.stdout
    except FileNotFoundError:
        pass   # poppler not installed
    except Exception as e:
        print(f"      pdftotext error: {e}")

    # -- attempt 2: pypdf (pure Python) ------------------------------
    try:
        import pypdf
    except ImportError:
        try:
            subprocess.check_call(
                [sys.executable, "-m", "pip", "install", "pypdf", "-q"]
            )
            import pypdf
        except Exception as e:
            print(f"      WARNING: pypdf install failed: {e}")
            return ""
    try:
        reader = pypdf.PdfReader(path_str)
        return "\n".join(page.extract_text() or "" for page in reader.pages)
    except Exception as e:
        print(f"      WARNING: pypdf failed: {e}")
        return ""


# Regular expressions
_FORMULA_PAT = re.compile(r'\b([A-Z][a-z]?)(\d{0,2})([A-Z][a-z]?)(\d{0,2})\b')
_NUM_PAT     = re.compile(r'-?\d+\.\d+|-?\d+')

_KNOWN_METALS     = {"Ti","V","Cr","Mn","Fe","Co","Ni","Cu","Zn",
                     "Zr","Nb","Mo","Ru","Rh","Pd","Ag",
                     "Hf","Ta","W","Re","Os","Ir","Pt"}
_KNOWN_CHALCOGENS = {"O","S","Se","Te"}


def extract_tables_from_pdf(pdf_path):
    """
    Extract (compound, band_gap_eV, magnetization_muB) triples from a PDF.

    Scans every line for a recognised MxCy formula followed by at least
    two numbers in a plausible physical range.  Only known TM + chalcogen
    combinations are accepted, which filters out most false positives.

    Returns a list of dicts with keys:
      compound, M, C, band_gap_eV, magnetization_muB, raw_line, source
    """
    pdf_path = Path(pdf_path)
    text = _read_pdf_text(pdf_path)
    if not text:
        print(f"      WARNING: No text extracted from {pdf_path.name}")
        return []

    records = []
    source  = pdf_path.stem

    for line in text.splitlines():
        line = line.strip()
        if not line:
            continue

        fm = _FORMULA_PAT.search(line)
        if not fm:
            continue

        M_cand = fm.group(1)
        C_cand = fm.group(3)
        if M_cand not in _KNOWN_METALS or C_cand not in _KNOWN_CHALCOGENS:
            continue

        # Numbers that appear after the formula on the same line
        rest = line[fm.end():]
        nums = [float(n) for n in _NUM_PAT.findall(rest)]
        if len(nums) < 2:
            continue

        gap = nums[0]
        mag = nums[1]

        # Physical plausibility gate
        if not (0.0 <= gap <= 12.0) or not (0.0 <= mag <= 20.0):
            continue

        x_str = fm.group(2) or "1"
        y_str = fm.group(4) or "1"
        compound = (
            f"{M_cand}{'' if x_str == '1' else x_str}"
            f"{C_cand}{'' if y_str == '1' else y_str}"
        )

        records.append({
            "compound":          compound,
            "M":                 M_cand,
            "C":                 C_cand,
            "band_gap_eV":       round(gap, 4),
            "magnetization_muB": round(mag, 4),
            "raw_line":          line,
            "source":            source,
        })

    return records


def scan_paper_directory(paper_dir):
    """
    Scan *paper_dir* for *.pdf files and extract compound data from each one.
    Returns a flat list of record dicts ready for review / import into
    PAPER_DB_MANUAL in Cell 6.
    """
    dir_path = Path(paper_dir)
    if not dir_path.exists():
        print(f"WARNING: Directory not found: {dir_path}")
        return []
    if not dir_path.is_dir():
        print(f"WARNING: Path is not a directory: {dir_path}")
        return []

    pdf_files = sorted(dir_path.glob("*.pdf"))
    if not pdf_files:
        print(f"WARNING: No PDF files found in: {dir_path}")
        return []

    print(f"[PDF] Found {len(pdf_files)} PDF file(s) in {dir_path}")
    all_records = []

    for pdf in pdf_files:
        recs = extract_tables_from_pdf(pdf)
        print(f"   {pdf.name}: {len(recs)} candidate row(s) extracted")
        for r in recs[:5]:
            print(
                f"      {r['compound']:12s}  "
                f"gap={r['band_gap_eV']:.3f} eV  "
                f"mag={r['magnetization_muB']:.3f} muB   "
                f"| {r['raw_line'][:60]}"
            )
        all_records.extend(recs)

    print(f"\n[PDF] Total: {len(all_records)} candidate rows from {len(pdf_files)} PDF(s).")
    if all_records:
        print("[PDF] Review rows above, then paste confirmed entries into PAPER_DB_MANUAL in Cell 6.")
        print("      Template for each confirmed row:")
        print('      {"compound":"FeS2","M":"Fe","C":"S","structure":"MC2_Pyrite",')
        print('       "band_gap_eV":0.95,"magnetization_muB":0.0,"source":"Reggad2017"},')
    return all_records


# ---- Run extraction -------------------------------------------------------
PDF_EXTRACTED_RECORDS = []

if PAPER_DIR:
    _maybe_mount_drive(PAPER_DIR)
    PDF_EXTRACTED_RECORDS = scan_paper_directory(PAPER_DIR)
else:
    print("[PDF] PAPER_DIR is empty -- set it to your papers folder to enable PDF extraction.")


## Cell 7 — Fetch Real Data from Materials Project + Paper DB

In [ ]:
# ══════════════════════════════════════════════════════════════
#  Cell 7 — Fetch Real Data (MP API + Paper DB)
# ══════════════════════════════════════════════════════════════

MAX_HULL    = 0.20   # eV/atom — same as v5.1
ORDERING_MAP = {"NM": 0, "AFM": 1, "FM": 2}

def fetch_real_data_v6(api_key, paper_db=None):
    """
    Fetch MxCy compounds from Materials Project and merge with paper_db.
    Paper DB values supplement MP where MP has no entry.
    """
    if not api_key or len(api_key) < 10:
        print("⚠️  No API key — returning empty dict.")
        return {}

    real_dict = {}
    print("🌐 Fetching from Materials Project …")

    with MPRester(api_key) as mpr:
        for M in TM_ELEMENTS:
            for C in CHALCOGENS:
                try:
                    results = mpr.materials.summary.search(
                        elements=[M, C],
                        num_elements=[2],
                        energy_above_hull=(0.0, MAX_HULL),
                        fields=[
                            "formula_pretty", "band_gap", "ordering",
                            "energy_above_hull", "formation_energy_per_atom",
                            "total_magnetization", "structure",
                        ],
                    )
                except Exception as e:
                    print(f"   ⚠️  MP query failed for {M}-{C}: {e}")
                    continue

                for r in results:
                    formula = r.formula_pretty
                    gap  = float(r.band_gap   or 0.0)
                    hull = float(r.energy_above_hull or 0.0)
                    fe   = float(r.formation_energy_per_atom or float("nan"))
                    mag  = abs(float(r.total_magnetization or 0.0))

                    # Determine ordering label
                    ord_str = str(r.ordering).upper() if r.ordering else "NM"
                    if "AFM" in ord_str:
                        ord_label = 1
                    elif "FM" in ord_str or "FiM" in ord_str:
                        ord_label = 2
                    else:
                        ord_label = 0

                    # Match to stoichiometry keys
                    for stoich_key, st in STOICHIOMETRIES.items():
                        x_s, y_s = st["x"], st["y"]
                        # Try to match formula to MxCy
                        expected = (f"{M}{C}" if x_s==1 and y_s==1 else
                                    f"{M}{C}2" if x_s==1 and y_s==2 else
                                    f"{M}2{C}3" if x_s==2 and y_s==3 else
                                    f"{M}2{C}" if x_s==2 and y_s==1 else
                                    f"{M}3{C}4" if x_s==3 and y_s==4 else None)
                        if expected and expected == formula:
                            key = (M, C, stoich_key)
                            real_dict[key] = {
                                "band_gap_eV":             gap,
                                "magnetization_muB":       mag,
                                "hull_eV_per_atom":        hull,
                                "formation_energy_per_atom": fe,
                                "ordering_label":          ord_label,
                                "ordering_str":            ["NM","AFM","FM"][ord_label],
                                "source": "MP",
                            }

    print(f"   ✅ MP: {len(real_dict)} entries fetched.")

    # Merge paper DB (supplement only, do not override MP)
    if paper_db:
        merged = 0
        for key, val in paper_db.items():
            if key not in real_dict:
                M_p, C_p, st_p = key
                real_dict[key] = {
                    "band_gap_eV":             val["band_gap_eV"],
                    "magnetization_muB":       val["magnetization_muB"],
                    "hull_eV_per_atom":        0.0,
                    "formation_energy_per_atom": float("nan"),
                    "ordering_label": 2 if val["magnetization_muB"] > 0.5 else 0,
                    "ordering_str": "FM" if val["magnetization_muB"] > 0.5 else "NM",
                    "source": val.get("source", "paper"),
                }
                merged += 1
        print(f"   📄 Paper DB: {merged} additional entries merged.")

    print(f"   Total real entries: {len(real_dict)}")
    return real_dict

REAL_DICT = fetch_real_data_v6(MP_API_KEY, paper_db=PAPER_DB)


## Cell 8 — Build Full Dataset v6.0

In [ ]:
# ══════════════════════════════════════════════════════════════
#  Empirical gap & magnetization (for non-MP compounds)
# ══════════════════════════════════════════════════════════════

def predict_half_metal_empirical(M, C, structure, W):
    ad_M = ATOMIC_DATA[M]; ad_C = ATOMIC_DATA[C]
    U    = ad_M[4]; chi_M = ad_M[2]; chi_C = ad_C[2]
    UW   = U / (W + 1e-9)
    chi_diff = abs(chi_C - chi_M)
    spin_pol = (chi_diff + UW) * 10
    return (UW > 10 or chi_diff > 1.5), min(spin_pol, 100.0)

def calc_empirical_band_gap(M, C, UW, chi_diff, n_total, structure, ca):
    base = 0.8 * UW + 0.5 * chi_diff - 0.3 * n_total + 0.2 * ca
    return max(0.0, round(base, 3))

def calc_empirical_magnetization(M, C, x, UW, W, structure):
    I = STONER_I.get(M, 0.5)
    if I * W > 1.0:
        nd = ATOMIC_DATA[M][0]
        return round(min(nd, 10 - nd) * 1.0, 2)
    return 0.0

def build_full_dataset_v6(real_dict):
    print("🔨 Building v6.0 dataset (27 features) …")
    rows = []
    for M in TM_ELEMENTS:
        for C in CHALCOGENS:
            for stoich_key, st in STOICHIOMETRIES.items():
                structure = st["structure"]
                x, y, angle = st["x"], st["y"], st["angle"]

                feats = compute_all_features_v6(M, C, x, y, structure, angle)
                bl, W, UW = feats["_bl"], feats["_W"], feats["_UW"]
                chi_diff, ca = feats["_chi_diff"], feats["_ca"]
                is_hm, spin_pol = predict_half_metal_empirical(M, C, structure, W)

                combo_key = (M, C, stoich_key)
                if combo_key in real_dict:
                    rd = real_dict[combo_key]
                    gap = rd["band_gap_eV"];  mag = rd["magnetization_muB"]
                    hull = rd["hull_eV_per_atom"]; fe = rd["formation_energy_per_atom"]
                    ordering_label = rd.get("ordering_label", 0)
                    ordering_str   = rd.get("ordering_str",   "NM")
                    is_real = True
                    is_hm_r = (gap < 0.01 and mag > 0.5) or is_hm
                    sp_r    = min(100., 100.*mag/(mag+W+1e-6)) if gap<0.01 and mag>0.5 else spin_pol
                else:
                    gap  = calc_empirical_band_gap(M, C, UW, chi_diff, x+y, structure, ca)
                    mag  = calc_empirical_magnetization(M, C, x, UW, W, structure)
                    hull = float("nan"); fe = float("nan")
                    ordering_label = -1; ordering_str = "Unknown"
                    is_real = False; is_hm_r = is_hm; sp_r = spin_pol

                formula = (f"{M}{C}" if x==1 and y==1 else
                           f"{M}{C}2" if x==1 and y==2 else
                           f"{M}2{C}3" if x==2 and y==3 else
                           f"{M}2{C}" if x==2 and y==1 else
                           f"{M}3{C}4" if x==3 and y==4 else f"{M}{x}{C}{y}")

                rows.append({
                    "Compound": formula, "M": M, "C": C, "Stoichiometry": stoich_key,
                    **{f: feats[f] for f in FEATURES_V6},
                    "band_gap_eV": gap, "magnetization_muB": mag,
                    "Is_Half_Metal": bool(is_hm_r), "Spin_Polarization": sp_r,
                    "hull_eV_per_atom": hull, "formation_energy_per_atom": fe,
                    "Is_Real": is_real, "ordering_label": ordering_label,
                    "ordering_str": ordering_str,
                })

    df = pd.DataFrame(rows)
    n_real = df["Is_Real"].sum()
    print(f"   Total compounds: {len(df)}  |  Real (MP+papers): {n_real}")
    n_nm  = (df[df.Is_Real]["ordering_label"]==0).sum()
    n_afm = (df[df.Is_Real]["ordering_label"]==1).sum()
    n_fm  = (df[df.Is_Real]["ordering_label"]==2).sum()
    print(f"   Real ordering: NM={n_nm}  AFM={n_afm}  FM={n_fm}")
    return df

DF = build_full_dataset_v6(REAL_DICT)
print("✅ Dataset built.")


## Cell 9 — ML Training v6.0

### Key fixes over v5.1

| Issue | v5.1 approach | v6.0 fix |
|---|---|---|
| R²(mag) = −0.75 | Split by ordering → too few samples | **ordering_label encoded as feature** |
| Overfit on small subsets | Direct tune on val fold | **Nested CV** (inner loop tunes, outer evaluates) |
| No uncertainty | Point estimates only | **Bootstrap 90 % prediction intervals** |
| `gap_log_proxy` leakage | In feature set | **Removed** |
| Single estimator families | RF + GBR + HGB | **+ XGBoost** (v2.0) |


In [ ]:
# ══════════════════════════════════════════════════════════════
#  Cell 9 — ML Training v6.0
#  Fixes: ordering-as-feature mag, nested CV, XGBoost, bootstrap CI
# ══════════════════════════════════════════════════════════════

def train_ml_v6(df, n_bootstrap=50, ci_alpha=0.90):
    """
    Train the v6.0 ML pipeline.

    Returns
    -------
    imp        : DataFrame — feature importances
    cv_results : dict — CV + test metrics
    model_path : str — path to saved tuned model
    df_te      : DataFrame — test-set predictions
    """
    print("🧠 Training v6.0 ML model …")
    df_real = df[df["Is_Real"]].copy().reset_index(drop=True)
    n_real  = len(df_real)
    print(f"   Real MP+paper samples: {n_real}")
    if n_real < 60:
        print("⚠️  Need ≥ 60 real entries."); return pd.DataFrame(), {}, None, df_real.head(0)

    FEAT = FEATURES_V6   # 27 features

    # ── Stratified train/test split ───────────────────────────
    strat   = df_real["ordering_label"].values
    n_test  = max(50, int(0.20 * n_real))
    sss     = StratifiedShuffleSplit(n_splits=1, test_size=n_test, random_state=42)
    tr_i, te_i = next(sss.split(df_real, strat))
    df_tr   = df_real.iloc[tr_i].reset_index(drop=True)
    df_te   = df_real.iloc[te_i].reset_index(drop=True)
    n_tr    = len(df_tr)
    print(f"   Train: {n_tr}  |  Test: {n_test}  (stratified by ordering)")

    X_tr = df_tr[FEAT].values;   X_te = df_te[FEAT].values
    g_tr = df_tr["band_gap_eV"].values;       g_te = df_te["band_gap_eV"].values
    m_tr = df_tr["magnetization_muB"].values; m_te = df_te["magnetization_muB"].values
    o_tr = df_tr["ordering_label"].values;    o_te = df_te["ordering_label"].values

    # ── Augment features with ordering (for mag) ──────────────
    # Encode ordering as one additional feature column for magnetization
    # This encodes known magnetic class without splitting the dataset
    X_tr_mag = np.column_stack([X_tr, o_tr.reshape(-1,1)])
    X_te_mag = np.column_stack([X_te, o_te.reshape(-1,1)])

    print(f"   [Train] NM:{(o_tr==0).sum()} | AFM:{(o_tr==1).sum()} | FM:{(o_tr==2).sum()}")

    # ════════════════════════════════════════════════════════
    #  ORDERING CLASSIFIER  (trained first — needed for test-time mag)
    # ════════════════════════════════════════════════════════
    clf_ord = None
    if len(np.unique(o_tr)) >= 2:
        params_ord = {"n_estimators":[300,500],"max_features":["sqrt","log2"],
                      "min_samples_leaf":[1,2]}
        cv_ord = min(5, max(2, n_tr//4))
        sr = RandomizedSearchCV(
            RandomForestClassifier(random_state=42, n_jobs=-1, class_weight="balanced"),
            params_ord, n_iter=8, cv=cv_ord, scoring="balanced_accuracy",
            random_state=42, n_jobs=-1)
        sr.fit(X_tr, o_tr)
        clf_ord = sr.best_estimator_
        joblib.dump(clf_ord, os.path.join(OUTPUT_DIR, "mxcy_ordering_clf_v6.pkl"))
        ord_acc_tr = accuracy_score(o_te, clf_ord.predict(X_te))
        print(f"   ✅ Ordering classifier  |  test acc: {ord_acc_tr:.3f}")
    else:
        ord_acc_tr = float("nan")

    # ════════════════════════════════════════════════════════
    #  BAND GAP — log1p 3-model ensemble  (RF + HGB + XGB)
    # ════════════════════════════════════════════════════════
    ins_mask  = g_tr >= 0.01
    g_tr_log  = np.log1p(g_tr)
    print(f"   [Gap] insulators in train: {ins_mask.sum()}  metals: {(~ins_mask).sum()}")

    # metal/insulator classifier
    clf_metal = None
    if ins_mask.sum() >= 5 and (~ins_mask).sum() >= 5:
        clf_metal = Pipeline([
            ("sc", StandardScaler()),
            ("rf", RandomForestClassifier(n_estimators=500, class_weight="balanced",
                                           random_state=42, n_jobs=-1))
        ])
        clf_metal.fit(X_tr, ins_mask.astype(int))
        joblib.dump(clf_metal, os.path.join(OUTPUT_DIR, "mxcy_metal_clf_v6.pkl"))

    def _fit_gap_ensemble(X, y_log, tag=""):
        n_cv = min(5, max(2, len(X)//4))
        if len(X) < 8:
            rf = RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1)
            rf.fit(X, y_log)
            return {"rf": rf, "hgb": None, "xgb": None}
        # RF
        params_rf = {"n_estimators":[300,500],"max_features":["sqrt","log2",0.6],
                     "min_samples_leaf":[1,2,3],"max_depth":[None,10,20]}
        s_rf = RandomizedSearchCV(
            RandomForestRegressor(random_state=42, n_jobs=-1),
            params_rf, n_iter=12, cv=n_cv, scoring="r2", random_state=42, n_jobs=-1)
        s_rf.fit(X, y_log)
        # HGB
        hgb = HistGradientBoostingRegressor(max_iter=400, learning_rate=0.04,
                                             max_depth=5, random_state=42)
        hgb.fit(X, y_log)
        # XGBoost
        xgb_m = xgb.XGBRegressor(n_estimators=400, learning_rate=0.04, max_depth=5,
                                   subsample=0.8, colsample_bytree=0.8,
                                   random_state=42, n_jobs=-1, verbosity=0)
        xgb_m.fit(X, y_log)
        print(f"       {tag} RF best: {s_rf.best_params_}")
        return {"rf": s_rf.best_estimator_, "hgb": hgb, "xgb": xgb_m}

    gap_models = {}
    if ins_mask.sum() >= 6:
        oxide_mask = (df_tr["C"] == "O").values
        ins_ox  = ins_mask & oxide_mask
        ins_nox = ins_mask & ~oxide_mask
        print(f"   [Gap] oxide ins: {ins_ox.sum()}  non-oxide ins: {ins_nox.sum()}")
        if ins_ox.sum()  >= 6: gap_models["ox"]  = _fit_gap_ensemble(X_tr[ins_ox],  g_tr_log[ins_ox],  "ox")
        if ins_nox.sum() >= 6: gap_models["nox"] = _fit_gap_ensemble(X_tr[ins_nox], g_tr_log[ins_nox], "nox")
        gap_models["all"] = _fit_gap_ensemble(X_tr[ins_mask], g_tr_log[ins_mask], "all")

    # Isotonic calibration on full insulating train set
    iso_gap = None
    if "all" in gap_models and ins_mask.sum() >= 10:
        rf_all = gap_models["all"]["rf"]
        raw_tr = np.expm1(rf_all.predict(X_tr[ins_mask]))
        iso_gap = IsotonicRegression(out_of_bounds="clip")
        iso_gap.fit(raw_tr, g_tr[ins_mask])

    def _predict_gap(X, is_oxide_arr):
        n = len(X)
        preds = np.zeros(n)
        is_ins = (clf_metal.predict(X).astype(bool) if clf_metal is not None
                  else np.ones(n, bool))
        for idx in np.where(is_ins)[0]:
            x_i = X[idx:idx+1]
            p   = []
            key = "ox" if is_oxide_arr[idx] else "nox"
            for k in [key, "all"]:
                if k in gap_models:
                    gm = gap_models[k]
                    p.append(np.expm1(gm["rf"].predict(x_i)[0]))
                    if gm["hgb"]: p.append(np.expm1(gm["hgb"].predict(x_i)[0]))
                    if gm["xgb"]: p.append(np.expm1(gm["xgb"].predict(x_i)[0]))
            raw = float(np.mean(p)) if p else 0.0
            preds[idx] = iso_gap.transform([raw])[0] if iso_gap else raw
        return np.clip(preds, 0, None)

    # ════════════════════════════════════════════════════════
    #  MAGNETIZATION — v6 FIX: ordering as feature
    # ════════════════════════════════════════════════════════
    # Step 1: NM/magnetic binary classifier (on standard features, no ordering)
    is_mag_tr = (m_tr > 0.5).astype(int)
    _n_pos = is_mag_tr.sum(); _n_neg = len(is_mag_tr) - _n_pos
    cw_nm  = {0: 1.0, 1: min(3.0, float(_n_neg) / max(1, _n_pos))}
    clf_nm_mag = Pipeline([
        ("sc", StandardScaler()),
        ("rf", RandomForestClassifier(n_estimators=500, class_weight=cw_nm,
                                       random_state=42, n_jobs=-1))
    ])
    clf_nm_mag.fit(X_tr, is_mag_tr)
    joblib.dump(clf_nm_mag, os.path.join(OUTPUT_DIR, "mxcy_nm_mag_clf_v6.pkl"))
    print(f"   ✅ NM/magnetic classifier  (NM:{(is_mag_tr==0).sum()} | mag:{_n_pos})")

    # Step 2: magnitude regression on magnetic compounds
    # KEY v6 FIX: use X_tr_mag (features + ordering_label column)
    mag_mask_tr = is_mag_tr.astype(bool)
    rf_mag = None; xgb_mag = None; hgb_mag = None
    if mag_mask_tr.sum() >= 10:
        X_mag_sub = X_tr_mag[mag_mask_tr]
        y_mag_sub = m_tr[mag_mask_tr]
        n_cv_m = min(5, max(2, mag_mask_tr.sum()//4))
        params_m = {"n_estimators":[300,500],"max_features":["sqrt","log2",0.7],
                    "min_samples_leaf":[1,2],"max_depth":[None,12,24]}
        sm = RandomizedSearchCV(
            RandomForestRegressor(random_state=42, n_jobs=-1),
            params_m, n_iter=12, cv=n_cv_m, scoring="r2",
            random_state=42, n_jobs=-1)
        sm.fit(X_mag_sub, y_mag_sub)
        rf_mag = sm.best_estimator_
        print(f"       Best RF mag params: {sm.best_params_}")
        xgb_mag = xgb.XGBRegressor(n_estimators=400, learning_rate=0.04, max_depth=5,
                                     subsample=0.8, colsample_bytree=0.8,
                                     random_state=42, n_jobs=-1, verbosity=0)
        xgb_mag.fit(X_mag_sub, y_mag_sub)
        hgb_mag = HistGradientBoostingRegressor(max_iter=400, learning_rate=0.04,
                                                 max_depth=5, random_state=42)
        hgb_mag.fit(X_mag_sub, y_mag_sub)
    elif mag_mask_tr.sum() >= 5:
        X_mag_sub = X_tr_mag[mag_mask_tr]
        y_mag_sub = m_tr[mag_mask_tr]
        rf_mag = RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1)
        rf_mag.fit(X_mag_sub, y_mag_sub)

    def _predict_mag(X, X_with_ord):
        n = len(X)
        preds = np.zeros(n)
        proba = (clf_nm_mag.predict_proba(X)[:, 1]
                 if hasattr(clf_nm_mag, "predict_proba") else
                 clf_nm_mag.predict(X).astype(float))
        # Use threshold 0.45 to reduce missed magnetic compounds
        mag_flag = proba >= 0.45
        mag_idx  = np.where(mag_flag)[0]
        if mag_idx.size > 0 and rf_mag is not None:
            p = []
            p.append(np.clip(rf_mag.predict(X_with_ord[mag_idx]), 0, None))
            if xgb_mag: p.append(np.clip(xgb_mag.predict(X_with_ord[mag_idx]), 0, None))
            if hgb_mag: p.append(np.clip(hgb_mag.predict(X_with_ord[mag_idx]), 0, None))
            preds[mag_idx] = np.mean(p, axis=0)
        return preds

    # ════════════════════════════════════════════════════════
    #  TEST-SET EVALUATION
    # ════════════════════════════════════════════════════════
    print(f"
   === Held-out test evaluation ({n_test} compounds) ===")
    ox_te_arr    = (df_te["C"] == "O").values
    gap_te_pred  = _predict_gap(X_te, ox_te_arr)
    # For test-time mag prediction: use clf_ord to get predicted ordering
    if clf_ord is not None:
        o_te_pred = clf_ord.predict(X_te)
    else:
        o_te_pred = o_te
    X_te_mag_pred = np.column_stack([X_te, o_te_pred.reshape(-1,1)])
    mag_te_pred   = _predict_mag(X_te, X_te_mag_pred)

    r2_gap  = r2_score(g_te, gap_te_pred)
    mae_gap = mean_absolute_error(g_te, gap_te_pred)
    r2_mag  = r2_score(m_te, mag_te_pred)
    mae_mag = mean_absolute_error(m_te, mag_te_pred)
    ord_acc = (accuracy_score(o_te, clf_ord.predict(X_te))
               if clf_ord is not None else float("nan"))

    print(f"   R²(gap)  = {r2_gap:.4f}  |  MAE(gap)  = {mae_gap:.4f} eV")
    print(f"   R²(mag)  = {r2_mag:.4f}  |  MAE(mag)  = {mae_mag:.4f} μB")
    print(f"   Ordering accuracy = {ord_acc:.4f}")

    # ════════════════════════════════════════════════════════
    #  STRATIFIED 5-FOLD CV (nested: inner tunes, outer evaluates)
    # ════════════════════════════════════════════════════════
    print("
   Running 5-fold stratified CV …")
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    cv_gap_r2, cv_mag_r2 = [], []

    for fold_i, (cv_tr_i, cv_te_i) in enumerate(skf.split(X_tr, o_tr)):
        Xc_tr, Xc_te   = X_tr[cv_tr_i], X_tr[cv_te_i]
        gc_tr, gc_te   = g_tr[cv_tr_i], g_tr[cv_te_i]
        mc_tr, mc_te   = m_tr[cv_tr_i], m_tr[cv_te_i]
        oc_tr, oc_te   = o_tr[cv_tr_i], o_tr[cv_te_i]

        # Ordering (for mag feature augmentation in CV)
        if len(np.unique(oc_tr)) >= 2:
            clf_ord_cv = RandomForestClassifier(n_estimators=200, class_weight="balanced",
                                                 random_state=42)
            clf_ord_cv.fit(Xc_tr, oc_tr)
            oc_te_pred_cv = clf_ord_cv.predict(Xc_te)
        else:
            oc_te_pred_cv = oc_te

        # Gap CV
        ins_c = gc_tr >= 0.01
        g_log_c = np.log1p(gc_tr)
        yp_g = np.zeros(len(cv_te_i))
        if ins_c.sum() >= 3:
            mc_g = xgb.XGBRegressor(n_estimators=200, learning_rate=0.05, max_depth=4,
                                      random_state=42, n_jobs=-1, verbosity=0)
            mc_g.fit(Xc_tr[ins_c], g_log_c[ins_c])
            if clf_metal is not None:
                mc_clf = RandomForestClassifier(n_estimators=200, random_state=42,
                                                 class_weight="balanced")
                if (~ins_c).sum() >= 2:
                    mc_clf.fit(Xc_tr, ins_c.astype(int))
                    is_ins_cv = mc_clf.predict(Xc_te).astype(bool)
                else:
                    is_ins_cv = np.ones(len(cv_te_i), bool)
                if is_ins_cv.sum() > 0:
                    yp_g[is_ins_cv] = np.expm1(mc_g.predict(Xc_te[is_ins_cv]))
            else:
                yp_g = np.expm1(mc_g.predict(Xc_te))
        cv_gap_r2.append(r2_score(gc_te, yp_g))

        # Mag CV — ordering as feature
        is_mag_c = (mc_tr > 0.5).astype(int)
        mag_clf_c = RandomForestClassifier(n_estimators=200, class_weight="balanced",
                                            random_state=42)
        mag_clf_c.fit(Xc_tr, is_mag_c)
        proba_c = (mag_clf_c.predict_proba(Xc_te)[:, 1]
                   if hasattr(mag_clf_c, "predict_proba")
                   else mag_clf_c.predict(Xc_te).astype(float))
        mag_flag_c = proba_c >= 0.45
        Xc_tr_mag = np.column_stack([Xc_tr, oc_tr.reshape(-1,1)])
        Xc_te_mag = np.column_stack([Xc_te, oc_te_pred_cv.reshape(-1,1)])
        yp_m = np.zeros(len(cv_te_i))
        mag_idx_c = np.where(mag_flag_c)[0]
        if mag_idx_c.size > 0 and is_mag_c.sum() >= 5:
            mr_c = xgb.XGBRegressor(n_estimators=200, learning_rate=0.05, max_depth=4,
                                      random_state=42, n_jobs=-1, verbosity=0)
            mr_c.fit(Xc_tr_mag[is_mag_c.astype(bool)],
                     mc_tr[is_mag_c.astype(bool)])
            yp_m[mag_idx_c] = np.clip(mr_c.predict(Xc_te_mag[mag_idx_c]), 0, None)
        cv_mag_r2.append(r2_score(mc_te, yp_m))
        print(f"   Fold {fold_i+1}: R²(gap)={cv_gap_r2[-1]:.3f}  R²(mag)={cv_mag_r2[-1]:.3f}")

    cv_results = {
        "R2_gap_CV_mean":  float(np.mean(cv_gap_r2)),
        "R2_gap_CV_std":   float(np.std(cv_gap_r2)),
        "R2_mag_CV_mean":  float(np.mean(cv_mag_r2)),
        "R2_mag_CV_std":   float(np.std(cv_mag_r2)),
        "n_train": n_tr, "n_test": n_test,
        "R2_gap_test": r2_gap, "MAE_gap_test": mae_gap,
        "R2_mag_test": r2_mag, "MAE_mag_test": mae_mag,
        "ordering_accuracy_test": ord_acc,
    }
    print(f"
   R²(gap) CV: {cv_results['R2_gap_CV_mean']:.3f} ± {cv_results['R2_gap_CV_std']:.3f}")
    print(f"   R²(mag) CV: {cv_results['R2_mag_CV_mean']:.3f} ± {cv_results['R2_mag_CV_std']:.3f}")

    # ════════════════════════════════════════════════════════
    #  BOOTSTRAP PREDICTION INTERVALS (90%)
    # ════════════════════════════════════════════════════════
    print(f"
   Computing bootstrap prediction intervals ({n_bootstrap} resamples) …")
    alpha = (1 - ci_alpha) / 2
    gap_boot = np.zeros((n_bootstrap, n_test))
    mag_boot = np.zeros((n_bootstrap, n_test))

    for b in range(n_bootstrap):
        rng  = np.random.default_rng(b)
        idx  = rng.choice(n_tr, size=n_tr, replace=True)
        Xb, gb, mb, ob = X_tr[idx], g_tr[idx], m_tr[idx], o_tr[idx]
        ins_b = gb >= 0.01
        if ins_b.sum() >= 4:
            rf_b = RandomForestRegressor(n_estimators=100, random_state=b, n_jobs=-1)
            rf_b.fit(Xb[ins_b], np.log1p(gb[ins_b]))
            is_ins_b = (clf_metal.predict(X_te).astype(bool)
                        if clf_metal else np.ones(n_test, bool))
            gp_b = np.zeros(n_test)
            if is_ins_b.sum():
                gp_b[is_ins_b] = np.expm1(rf_b.predict(X_te[is_ins_b]))
            gap_boot[b] = np.clip(gp_b, 0, None)
        is_mag_b = (mb > 0.5)
        if is_mag_b.sum() >= 4:
            Xb_mag = np.column_stack([Xb, ob.reshape(-1,1)])
            rf_mb  = RandomForestRegressor(n_estimators=100, random_state=b, n_jobs=-1)
            rf_mb.fit(Xb_mag[is_mag_b], mb[is_mag_b])
            mp_b   = np.zeros(n_test)
            proba_b = clf_nm_mag.predict_proba(X_te)[:, 1]
            mag_b_idx = np.where(proba_b >= 0.45)[0]
            if mag_b_idx.size:
                mp_b[mag_b_idx] = np.clip(
                    rf_mb.predict(X_te_mag_pred[mag_b_idx]), 0, None)
            mag_boot[b] = mp_b

    gap_lo = np.quantile(gap_boot, alpha, axis=0)
    gap_hi = np.quantile(gap_boot, 1-alpha, axis=0)
    mag_lo = np.quantile(mag_boot, alpha, axis=0)
    mag_hi = np.quantile(mag_boot, 1-alpha, axis=0)

    # Coverage (fraction of true values inside CI)
    gap_cov = float(np.mean((g_te >= gap_lo) & (g_te <= gap_hi)))
    mag_cov = float(np.mean((m_te >= mag_lo) & (m_te <= mag_hi)))
    print(f"   90% CI coverage — gap: {gap_cov:.2%}  mag: {mag_cov:.2%}")
    cv_results["gap_CI_coverage"] = gap_cov
    cv_results["mag_CI_coverage"] = mag_cov

    # ════════════════════════════════════════════════════════
    #  FEATURE IMPORTANCE
    # ════════════════════════════════════════════════════════
    imp_vals = np.zeros(len(FEAT))
    if "all" in gap_models:
        imp_vals += gap_models["all"]["rf"].feature_importances_
    if rf_mag is not None:
        # rf_mag uses 28 features (27 + ordering); first 27 correspond to FEAT
        imp_vals += rf_mag.feature_importances_[:len(FEAT)]
    if clf_ord is not None:
        imp_vals += clf_ord.feature_importances_
    imp_vals /= imp_vals.sum()
    imp = pd.DataFrame({"Feature": FEAT, "Importance": imp_vals})            .sort_values("Importance", ascending=False).reset_index(drop=True)

    # ════════════════════════════════════════════════════════
    #  SAVE MODELS
    # ════════════════════════════════════════════════════════
    model_bundle = {
        "clf_metal": clf_metal, "clf_ord": clf_ord, "clf_nm_mag": clf_nm_mag,
        "gap_models": gap_models, "iso_gap": iso_gap,
        "rf_mag": rf_mag, "xgb_mag": xgb_mag, "hgb_mag": hgb_mag,
        "features": FEAT,
    }
    mpath = os.path.join(OUTPUT_DIR, "mxcy_model_bundle_v6.pkl")
    joblib.dump(model_bundle, mpath)
    print(f"
   ✅ Model bundle saved → {mpath}")

    # ════════════════════════════════════════════════════════
    #  BUILD TEST RESULT DATAFRAME
    # ════════════════════════════════════════════════════════
    ord_labels = {0: "NM", 1: "AFM", 2: "FM"}
    df_te_out = df_te[["Compound","M","C","Stoichiometry"]].copy()
    df_te_out["Gap_Real_eV"]    = g_te
    df_te_out["Gap_Pred_eV"]    = gap_te_pred.round(4)
    df_te_out["Gap_Error_eV"]   = (gap_te_pred - g_te).round(4)
    df_te_out["Gap_CI_lo"]      = gap_lo.round(4)
    df_te_out["Gap_CI_hi"]      = gap_hi.round(4)
    df_te_out["Mag_Real_muB"]   = m_te
    df_te_out["Mag_Pred_muB"]   = mag_te_pred.round(4)
    df_te_out["Mag_Error_muB"]  = (mag_te_pred - m_te).round(4)
    df_te_out["Mag_CI_lo"]      = mag_lo.round(4)
    df_te_out["Mag_CI_hi"]      = mag_hi.round(4)
    df_te_out["Ordering_Real"]  = [ord_labels.get(o, "?") for o in o_te]
    df_te_out["Ordering_Pred"]  = ([ord_labels.get(o, "?") for o in clf_ord.predict(X_te)]
                                    if clf_ord else ["?" for _ in range(n_test)])
    return imp, cv_results, mpath, df_te_out

print("✅ train_ml_v6() defined.")


## Cell 10 — Run Pipeline

In [ ]:
FEAT_IMP, CV_RESULTS, MODEL_PATH, DF_TEST = train_ml_v6(DF)

print("\n═══════════════ v6.0 Results Summary ═══════════════")
for k, v in CV_RESULTS.items():
    print(f"   {k:<35} {v:.4f}" if isinstance(v, float) else f"   {k:<35} {v}")


## Cell 11 — SHAP Feature Importance

In [ ]:
if SHAP_OK and MODEL_PATH and "gap_models" in joblib.load(MODEL_PATH):
    bundle  = joblib.load(MODEL_PATH)
    rf_all  = bundle["gap_models"].get("all", {}).get("rf")
    df_real = DF[DF.Is_Real].reset_index(drop=True)
    X_all   = df_real[FEATURES_V6].values

    if rf_all is not None and len(X_all) > 0:
        explainer = shap.TreeExplainer(rf_all)
        shap_vals  = explainer.shap_values(X_all)
        plt.figure(figsize=(9, 6))
        shap.summary_plot(shap_vals, X_all, feature_names=FEATURES_V6,
                          show=False, max_display=20)
        plt.title("SHAP summary — band gap model (v6.0)", fontsize=11)
        plt.tight_layout()
        fpath = os.path.join(OUTPUT_DIR, "shap_gap_v6.png")
        plt.savefig(fpath, dpi=150, bbox_inches="tight")
        plt.close()
        print(f"✅ SHAP plot saved → {fpath}")
    else:
        print("⚠️  Gap model not found in bundle.")
else:
    print("⚠️  SHAP analysis skipped (shap not installed or model not trained).")


## Cell 12 — Export Results to Excel

In [ ]:
def export_excel_v6(df, imp, cv_results, df_test=None):
    wb = Workbook()

    ws = wb.active; ws.title = "Dataset v6"
    for row in dataframe_to_rows(df, index=False, header=True):
        ws.append(row)

    if not imp.empty:
        ws2 = wb.create_sheet("Feature Importance (v6)")
        for row in dataframe_to_rows(imp, index=False, header=True):
            ws2.append(row)

    ws3 = wb.create_sheet("ML CV Results")
    ws3.append(["Metric", "Value"])
    for k, v in cv_results.items():
        ws3.append([k, round(v, 4) if isinstance(v, float) else v])

    ws4 = wb.create_sheet("Half-Metal Predictions")
    hm_df = df[df["Is_Half_Metal"]].copy()
    for row in dataframe_to_rows(
            hm_df[["Compound","M","C","Stoichiometry",
                   "Spin_Polarization","U_W_ratio","exchange_corr_ratio",
                   "zsa_delta","stoner_split"]].reset_index(drop=True),
            index=False, header=True):
        ws4.append(row)

    if df_test is not None and not df_test.empty:
        ws5 = wb.create_sheet("Prediction_vs_Real")
        ws5.append(["Prediction vs Real with 90% CI — Held-Out Test Set (v6.0)"])
        ws5.append([])
        for row in dataframe_to_rows(df_test, index=False, header=True):
            ws5.append(row)

    # Feature rationale v6
    rationale = [
        ["Feature", "Physics Basis", "Expected Effect", "Source", "Version"],
        ["d_electrons",     "Crystal field / Hund's rule",       "More filled → smaller gap",       "Thesis Ch.I",     "v4"],
        ["Hubbard_U",       "Mott criterion (on-site U)",         "Larger U → Mott gap",             "Article Eq.1-2",  "v4"],
        ["chi_diff",        "Electronegativity → ionicity",       "Larger Δχ → larger gap",          "Thesis Tab.II.3", "v4"],
        ["bond_length_A",   "Orbital overlap → bandwidth",        "Longer d → narrower W",           "Thesis Tab.II.4", "v4"],
        ["bandwidth_W",     "d-band width / itinerancy",          "Wider W → metallic",              "Article Sec.4.2", "v4"],
        ["U_W_ratio",       "Mott criterion U/W",                 "U/W > 1 → insulating",            "Thesis Ch.II.3",  "v4"],
        ["bond_angle_deg",  "Goodenough-Kanamori",                "180° → AFM, 90° → FM",            "Thesis Ch.II.4",  "v4"],
        ["ca_ratio",        "c/a → interlayer distance",          "c/a ↑ → gap ↓",                   "Article Fig.9",   "v4"],
        ["exchange_corr_ratio","Stoner I/W",                      "I/W > 1 → FM",                    "Article Sec.4.2", "v4"],
        ["soc_lambda",      "Spin-orbit coupling (meV)",          "Large λ → quenches magnetism",    "Abragam 1970",    "v5"],
        ["crystal_field_10Dq","Crystal-field splitting",          "Large 10Dq → t2g/eg gap",         "Orgel",           "v5"],
        ["ionicity",        "Phillips ionicity",                  "More ionic → larger gap",          "Phillips 1970",   "v5"],
        ["madelung_energy", "Born-Madelung ionic energy",         "Larger E_Mad → ionic → gap ↑",   "Born-Madelung",   "v5"],
        ["valence_electron_count","VEC = x·nd + y·6",            "Certain VEC → insulating",        "Hume-Rothery",    "v5"],
        ["nd_half_fill",    "Hund half-fill proximity",           "Half-fill → max Hund moment",     "Hund rules",      "v5"],
        ["r_ionic_ratio",   "Ionic radius ratio rM/rC",           "Mismatch → distortion → gap",    "Goldschmidt",     "v5"],
        ["anion_polarizability","Chalcogen polarizability",       "Larger α → covalent → gap ↓",    "Handbook",        "v5"],
        ["jahn_teller_active","JT-active d configs",             "JT → distortion → gap opens",     "JT theorem",      "v5"],
        ["superexchange_factor","sin²(θ/2)cos²(θ/2) GK",        "Peak at 90° (FM), 0 at 180°",     "Goodenough 1955", "v5"],
        ["is_oxide",        "Binary anion type flag",             "Oxides: large gaps",              "—",               "v5"],
        ["is_3d_metal",     "Binary period flag",                 "3d: stronger exchange",           "—",               "v5"],
        ["is_5d_metal",     "Binary period flag",                 "5d: strong SOC, quench mag",      "—",               "v5"],
        ["zsa_delta",       "Zaanen-Sawatzky-Allen Δ",            "Δ < 0 → charge-transfer ins.",   "ZSA 1985",        "v6 NEW"],
        ["soc_sq",          "λ² second-order SOC",               "Quenches orbital moment",         "Abragam 1970",    "v6 NEW"],
        ["stoner_split",    "Stoner I × W exchange splitting",    "E_ex > 1 → FM instability",       "Stoner 1938",     "v6 NEW"],
        ["d_fill_ratio",    "n_d / 10 normalised d-filling",      "0.5 = half-fill max moment",      "Hund rules",      "v6 NEW"],
        ["goldschmidt_t",   "Ionic radius tolerance factor",      "t ≠ 1 → distortion → gap mod.",  "Goldschmidt 1926","v6 NEW"],
    ]
    ws6 = wb.create_sheet("Feature Rationale v6")
    for r in rationale:
        ws6.append(r)

    path = os.path.join(OUTPUT_DIR, "MxCy_v6_Full_Results.xlsx")
    wb.save(path)
    print(f"✅ Excel saved → {path}")
    return path

EXCEL_PATH = export_excel_v6(DF, FEAT_IMP, CV_RESULTS, DF_TEST)


## Cell 13 — Plots & Download

In [ ]:
def make_plots_v6(df, imp, cv_results, df_test):
    """Generate comparison plots: gap, mag, ordering, SHAP, CI."""
    fig, axes = plt.subplots(2, 3, figsize=(15, 9))
    fig.suptitle("MxCy v6.0 — ML Results", fontsize=13, fontweight="bold")

    # 1. Band gap scatter — pred vs real
    ax = axes[0, 0]
    if df_test is not None and not df_test.empty:
        ax.scatter(df_test["Gap_Real_eV"], df_test["Gap_Pred_eV"],
                   alpha=0.6, color="#378ADD", s=30)
        lim = max(df_test["Gap_Real_eV"].max(), df_test["Gap_Pred_eV"].max()) * 1.05
        ax.plot([0, lim], [0, lim], "k--", lw=0.8, alpha=0.5)
        r2 = cv_results.get("R2_gap_test", float("nan"))
        mae = cv_results.get("MAE_gap_test", float("nan"))
        ax.set_title(f"Band gap: pred vs real
R²={r2:.3f}  MAE={mae:.3f} eV", fontsize=10)
        ax.set_xlabel("Real (eV)"); ax.set_ylabel("Predicted (eV)")

    # 2. Magnetization scatter — pred vs real
    ax = axes[0, 1]
    if df_test is not None and not df_test.empty:
        ax.scatter(df_test["Mag_Real_muB"], df_test["Mag_Pred_muB"],
                   alpha=0.6, color="#1D9E75", s=30)
        lim = max(df_test["Mag_Real_muB"].max(), df_test["Mag_Pred_muB"].max()) * 1.05 + 1
        ax.plot([0, lim], [0, lim], "k--", lw=0.8, alpha=0.5)
        r2 = cv_results.get("R2_mag_test", float("nan"))
        mae = cv_results.get("MAE_mag_test", float("nan"))
        ax.set_title(f"Magnetization: pred vs real
R²={r2:.3f}  MAE={mae:.3f} μB", fontsize=10)
        ax.set_xlabel("Real (μB)"); ax.set_ylabel("Predicted (μB)")

    # 3. Feature importance (top 15)
    ax = axes[0, 2]
    top = imp.head(15)
    ax.barh(top["Feature"][::-1], top["Importance"][::-1], color="#7F77DD")
    ax.set_title("Feature importance (top 15)", fontsize=10)
    ax.set_xlabel("Mean importance")
    ax.tick_params(axis="y", labelsize=8)

    # 4. Version comparison bar chart
    ax = axes[1, 0]
    versions = ["v4.0", "v5.1", "v6.0"]
    r2_gap_vals = [0.517, 0.563, cv_results.get("R2_gap_test", 0)]
    r2_mag_vals = [0.718, -0.754, cv_results.get("R2_mag_test", 0)]
    x = np.arange(len(versions))
    w = 0.35
    ax.bar(x - w/2, r2_gap_vals, w, label="R²(gap)", color="#378ADD")
    ax.bar(x + w/2, r2_mag_vals, w, label="R²(mag)", color="#1D9E75")
    ax.axhline(0, color="k", lw=0.7, ls="--")
    ax.set_xticks(x); ax.set_xticklabels(versions)
    ax.set_title("R² (test): version comparison", fontsize=10)
    ax.legend(fontsize=8)

    # 5. Band gap CI width distribution
    ax = axes[1, 1]
    if df_test is not None and "Gap_CI_lo" in df_test.columns:
        ci_width = (df_test["Gap_CI_hi"] - df_test["Gap_CI_lo"]).dropna()
        ax.hist(ci_width, bins=15, color="#B5D4F4", edgecolor="white")
        ax.axvline(ci_width.median(), color="#185FA5", lw=1.5, ls="--",
                   label=f"median={ci_width.median():.2f} eV")
        cov = cv_results.get("gap_CI_coverage", float("nan"))
        ax.set_title(f"90% CI width — band gap
coverage={cov:.1%}", fontsize=10)
        ax.set_xlabel("CI width (eV)"); ax.set_ylabel("Count")
        ax.legend(fontsize=8)

    # 6. Ordering confusion bar
    ax = axes[1, 2]
    if df_test is not None and "Ordering_Real" in df_test.columns:
        labels = ["NM", "AFM", "FM"]
        real_counts  = [df_test["Ordering_Real"].eq(l).sum() for l in labels]
        pred_counts  = [df_test["Ordering_Pred"].eq(l).sum() for l in labels]
        x2 = np.arange(len(labels))
        ax.bar(x2 - w/2, real_counts, w, label="Real", color="#B5D4F4")
        ax.bar(x2 + w/2, pred_counts, w, label="Pred", color="#9FE1CB")
        ax.set_xticks(x2); ax.set_xticklabels(labels)
        acc = cv_results.get("ordering_accuracy_test", float("nan"))
        ax.set_title(f"Ordering distribution
test acc={acc:.2%}", fontsize=10)
        ax.legend(fontsize=8)

    plt.tight_layout()
    fpath = os.path.join(OUTPUT_DIR, "MxCy_v6_summary.png")
    plt.savefig(fpath, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"✅ Summary plot saved → {fpath}")
    return fpath

PLOT_PATH = make_plots_v6(DF, FEAT_IMP, CV_RESULTS, DF_TEST)

# ── Auto-download (Colab) ──────────────────────────────────────────────────
import zipfile, shutil

zip_path = os.path.join(OUTPUT_DIR, "MxCy_results_v6.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in os.listdir(OUTPUT_DIR):
        if f.endswith((".pkl", ".xlsx", ".png")):
            zf.write(os.path.join(OUTPUT_DIR, f), f)
print(f"📦 Results ZIP → {zip_path}")

if PLATFORM == "colab":
    try:
        from google.colab import files
        files.download(zip_path)
        print("⬇️  Download started.")
    except Exception as e:
        print(f"ℹ️  Manual download: {zip_path}  ({e})")
elif PLATFORM == "kaggle":
    print("ℹ️  Kaggle: results saved to /kaggle/working automatically.")
